In [1]:
import numpy as np
import pandas as pd


In [2]:
# Required dataset loading (compact)
import torch

# -------------------------
# Device configuration
# -------------------------
# Options: 'auto', 'cpu', 'cuda'
DEVICE_MODE = 'auto'

if DEVICE_MODE == 'cpu':
    device = torch.device('cpu')
elif DEVICE_MODE == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError("DEVICE_MODE='cuda' but CUDA is not available.")
    device = torch.device('cuda')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# NSL-KDD fresh load (keeps original attack labels for grouping)
nsl_columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
    'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'attack', 'level'
 ]

nsl_train = pd.read_csv('./dataset/nslkdd/KDDTrain+.txt', header=None)
nsl_test = pd.read_csv('./dataset/nslkdd/KDDTest+.txt', header=None)
nsl_train.columns = nsl_columns
nsl_test.columns = nsl_columns
nsl_kdd_fresh = pd.concat([nsl_train, nsl_test], ignore_index=True)

# CICIDS-2017 and UAVIDS
cicd2017 = pd.read_csv('./dataset/cicids/Wednesday-workingHours.pcap_ISCX.csv')
uavids = pd.read_csv('./dataset/uavids/UAVIDS-2025.csv')

# Shared dataset registry used by HPO + experiments
datasets = {
    'CICIDS2017': cicd2017,
    'NSL-KDD': nsl_kdd_fresh,
    'UAVIDS': uavids,
}

target_columns = {
    'NSL-KDD': 'attack',
    'CICIDS2017': ' Label',
    'UAVIDS': 'label',
}

print('Loaded datasets:', {k: v.shape for k, v in datasets.items()})
print(f"Selected device mode: {DEVICE_MODE}")
print(f'Using device: {device}')

Loaded datasets: {'CICIDS2017': (692703, 85), 'NSL-KDD': (148517, 43), 'UAVIDS': (122171, 23)}
Selected device mode: auto
Using device: cuda


# Utils and Pipeline Functions

In [3]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

def preprocess_data(dataframe):
    df = pd.DataFrame(dataframe)
    numeric_cols = df.select_dtypes(include=['number']).columns
    categorical_cols = df.select_dtypes(exclude=['number']).columns

    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=numeric_cols)
    df = df.dropna(subset=categorical_cols)

    if not categorical_cols.empty:
        df[categorical_cols] = df[categorical_cols].astype(str)
        for col in categorical_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])

    return df

!pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
!pip install torch-geometric

In [4]:


import torch
from torch_geometric.data import Data

c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(
c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:124: UserWarning: An issue occurred while importing 'torch-

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=1):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bigru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=4,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = x.unsqueeze(1)
        x, _ = self.bigru(x)
        x = x[:, -1, :]
        x = self.classifier(x)
        return x.squeeze(-1) if x.size(-1) == 1 else x

In [6]:
def run_all_benchmarks(train_X, test_X, train_y, test_y, train_graph, test_graph, k=10, class_weights=None, model_hparams=None):
    import os, gc, time, tracemalloc
    from sklearn.preprocessing import LabelBinarizer
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, roc_curve, classification_report, confusion_matrix
    )

    results = []
    class_names = train_y.unique()

    lb = LabelBinarizer()
    lb.fit(train_y)

    input_dim = train_graph.x.shape[1]
    output_dim = 1 if len(train_graph.y.unique()) == 2 else len(train_graph.y.unique())

    model_hparams = model_hparams or {}
    hidden_dim = int(model_hparams.get('hidden_dim', 64))
    learning_rate = float(model_hparams.get('lr', 0.001))
    weight_decay = float(model_hparams.get('weight_decay', 0.0))
    train_epochs = int(model_hparams.get('epochs', 300))

    gnn_models = {
        "BG-GCN": GCN(input_dim, hidden_dim=hidden_dim, output_dim=output_dim)
    }

    confusion_matrices = {}
    split_metrics_tables = {}

    def evaluate_split(logits, labels, class_count):
        labels_np = labels.detach().cpu().numpy()

        if class_count == 2:
            loss_val = F.binary_cross_entropy_with_logits(
                logits.squeeze(),
                labels.float()
            ).item()
            probas_pos = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            preds = (probas_pos > 0.5).astype(int)
        else:
            loss_val = F.cross_entropy(logits, labels.long()).item()
            probas = torch.softmax(logits, dim=1).detach().cpu().numpy()
            preds = probas.argmax(axis=1)

        return {
            "Loss": loss_val,
            "Accuracy": accuracy_score(labels_np, preds),
            "Precision": precision_score(labels_np, preds, average='macro', zero_division=0),
            "Recall": recall_score(labels_np, preds, average='macro', zero_division=0),
            "F1 Score": f1_score(labels_np, preds, average='macro', zero_division=0),
        }

    for name, model in gnn_models.items():
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        gc.collect()
        torch.cuda.empty_cache()
        print(f"\n--- Training {name} ---")
        model = model.to(device)
        tracemalloc.start()
        start_time = time.time()

        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, train_epochs))
        weights_for_loss = None
        if class_weights is not None and len(class_names) > 2:
            weights_for_loss = class_weights.to(device)
            if weights_for_loss.numel() != output_dim:
                weights_for_loss = None

        model.train()
        train_start = time.perf_counter()
        for _ in range(train_epochs):
            optimizer.zero_grad()
            out_train = model(train_graph.x, train_graph.edge_index)
            if len(class_names) == 2:
                loss = F.binary_cross_entropy_with_logits(
                    out_train[train_graph.train_mask].squeeze(),
                    train_graph.y[train_graph.train_mask].float()
                )
            else:
                loss = F.cross_entropy(
                    out_train[train_graph.train_mask].squeeze(),
                    train_graph.y[train_graph.train_mask].long(),
                    weight=weights_for_loss
                )
            loss.backward()
            optimizer.step()
            scheduler.step()
        train_time = time.perf_counter() - train_start

        model.eval()
        with torch.no_grad():
            out = model(test_graph.x, test_graph.edge_index)

            if len(class_names) == 2:
                probas = torch.sigmoid(out).cpu().numpy()
                if probas.ndim > 1 and probas.shape[1] == 1:
                    probas_pos = probas[:, 0]
                else:
                    probas_pos = probas.ravel()
                pred = (probas_pos > 0.5).astype(int)
            else:
                probas = torch.softmax(out, dim=1).cpu().numpy()
                pred = probas.argmax(axis=1)

        if len(class_names) == 2:
            y_true = np.asarray(test_y)
            y_pred = np.asarray(pred).ravel()
            y_proba_for_auc = np.asarray(probas_pos).ravel()
        else:
            y_true = np.asarray(test_y)
            y_pred = np.asarray(pred).ravel()
            y_proba_for_auc = np.asarray(probas)

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        confusion = confusion_matrix(y_true, y_pred)
        confusion_matrices[name] = confusion

        auc = None
        try:
            if len(class_names) == 2:
                auc = roc_auc_score(y_true, y_proba_for_auc)
            else:
                auc = roc_auc_score(y_true, y_proba_for_auc, multi_class='ovr', average='macro')
                y_b = lb.transform(y_true)
                for i in range(len(class_names)):
                    _ = roc_curve(y_b[:, i], y_proba_for_auc[:, i])
        except ValueError:
            auc = None

        with torch.no_grad():
            train_forward_start = time.perf_counter()
            out_full_train = model(train_graph.x, train_graph.edge_index)
            train_forward_time = time.perf_counter() - train_forward_start

            test_forward_start = time.perf_counter()
            out_full_test = model(test_graph.x, test_graph.edge_index)
            test_forward_time = time.perf_counter() - test_forward_start

        train_mask = train_graph.train_mask
        val_mask = ~train_mask

        split_rows = []
        if train_mask.sum().item() > 0:
            train_eval_start = time.perf_counter()
            train_metrics = evaluate_split(
                out_full_train[train_mask],
                train_graph.y[train_mask],
                len(class_names),
            )
            train_eval_time = time.perf_counter() - train_eval_start
            split_rows.append({
                "Split": "Trainset",
                "Train Loop Time (s)": train_time,
                "Forward Time (s)": train_forward_time,
                "Eval Time (s)": train_eval_time,
                "Computation Time (s)": train_time + train_forward_time + train_eval_time,
                **train_metrics,
            })

        if val_mask.sum().item() > 0:
            val_eval_start = time.perf_counter()
            val_metrics = evaluate_split(
                out_full_train[val_mask],
                train_graph.y[val_mask],
                len(class_names),
            )
            val_eval_time = time.perf_counter() - val_eval_start
            split_rows.append({
                "Split": "validation set",
                "Train Loop Time (s)": 0.0,
                "Forward Time (s)": 0.0,
                "Eval Time (s)": val_eval_time,
                "Computation Time (s)": val_eval_time,
                **val_metrics,
            })

        test_eval_start = time.perf_counter()
        test_metrics = evaluate_split(
            out_full_test,
            test_graph.y,
            len(class_names),
        )
        test_eval_time = time.perf_counter() - test_eval_start
        split_rows.append({
            "Split": "Test set",
            "Train Loop Time (s)": 0.0,
            "Forward Time (s)": test_forward_time,
            "Eval Time (s)": test_eval_time,
            "Computation Time (s)": test_forward_time + test_eval_time,
            **test_metrics,
        })

        split_df = pd.DataFrame(split_rows)
        split_df = split_df[["Split", "Train Loop Time (s)", "Forward Time (s)", "Eval Time (s)", "Computation Time (s)", "Loss", "Accuracy", "Precision", "Recall", "F1 Score"]]
        split_metrics_tables[name] = split_df

        end_time = time.time()
        mem_consumption = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()

        results.append({
            "Model": name,
            "Accuracy": accuracy,
            "AUC": auc,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Classification Report": report,
            "Confusion Matrix": confusion,
            "Time (s)": f"{end_time - start_time:.2f} s",
            "Memory (MB)": f"{mem_consumption / 1e6:.2f} MB"
        })

        out = out.cpu() if isinstance(out, torch.Tensor) else out
        del out, out_full_train, out_full_test
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results).drop(columns=["Classification Report", "Confusion Matrix"])
    results_df = results_df.sort_values("Accuracy", ascending=False)
    print("\nBenchmark Results:")
    print(results_df.to_markdown(index=False))

    return results_df, confusion_matrices, split_metrics_tables


In [7]:
# Graph helper functions
def adaptive_graph_construction(X, y, adaptive_metric='euclidean', threshold=0.5):
    from sklearn.metrics.pairwise import pairwise_distances
    from sklearn.neighbors import kneighbors_graph
    from torch_geometric.data import Data

    distances = pairwise_distances(X, metric=adaptive_metric)
    adj = (distances < threshold).astype(int)
    adj = kneighbors_graph(X, 20, metric='euclidean').toarray() * adj
    edge_index = torch.tensor(adj.nonzero(), dtype=torch.long)
    features = torch.tensor(X, dtype=torch.float)
    labels = torch.tensor(y, dtype=torch.long)
    return Data(x=features, edge_index=edge_index, y=labels)

## Step 1: Hyperparameter Optimization and Before/After Comparison
Run the next cell to execute each dataset one-by-one in this order:
1. Experiment with default BG-GCN parameters (without HPO)
2. HPO (Secretary Bird Optimization)
3. Experiment with tuned parameters (after HPO)

Configuration is inside the pipeline cell and dataset-loading cell:
- `NUM_DATASETS_TO_TEST = 1`, `2`, or `'all'`
- `RUN_HPO_IN_THIS_PIPELINE = True/False`
- `HPO_CFG` and `EXPERIMENT_CFG` for budget and sampling
- `DEVICE_MODE = 'auto' | 'cpu' | 'cuda'` (in Cell 2)

In [8]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


def _map_nsl_attack_group(label):
    dos_attacks = {
        'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop',
        'apache2', 'mailbomb', 'processtable', 'udpstorm'
    }
    probing_attacks = {'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint'}
    u2r_attacks = {'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack', 'xterm', 'ps'}
    r2l_attacks = {
        'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy',
        'warezclient', 'warezmaster', 'httptunnel', 'named', 'sendmail',
        'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop'
    }

    label = str(label).strip()
    if label == 'normal':
        return 'normal'
    if label in probing_attacks:
        return 'probing'
    if label in dos_attacks:
        return 'DoS'
    if label in u2r_attacks:
        return 'U2R'
    if label in r2l_attacks:
        return 'R2L'
    return np.nan


def prepare_dataset_for_sbo(dataset_name, datasets_dict, target_columns_dict, sample_size=6000):
    target_col = target_columns_dict[dataset_name]
    temp = datasets_dict[dataset_name].copy()
    original_target = temp[target_col].copy()

    df = preprocess_data(temp)
    df[target_col] = original_target.iloc[:len(df)].reset_index(drop=True)

    if dataset_name == 'NSL-KDD':
        df[target_col] = df[target_col].apply(_map_nsl_attack_group)

    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    if len(df) > sample_size:
        class_counts = df[target_col].value_counts()
        sampled_parts = []
        for class_label, count in class_counts.items():
            class_df = df[df[target_col] == class_label]
            n_take = max(1, int(round(sample_size * count / len(df))))
            n_take = min(n_take, len(class_df))
            sampled_parts.append(class_df.sample(n=n_take, random_state=42))
        df = pd.concat(sampled_parts, ignore_index=True)

    y_raw = df[target_col].astype(str)
    X = df.drop(columns=[target_col])

    numeric_cols = X.select_dtypes(include=['number']).columns
    scaler = StandardScaler()
    X = pd.DataFrame(scaler.fit_transform(X[numeric_cols]))

    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y_raw))

    class_counts = y.value_counts()
    valid_classes = class_counts[class_counts >= 3].index
    if len(valid_classes) < class_counts.shape[0]:
        keep_mask = y.isin(valid_classes)
        X = X.loc[keep_mask].reset_index(drop=True)
        y = y.loc[keep_mask].reset_index(drop=True)

    if y.nunique() < 2:
        raise ValueError(f'Not enough classes for SBO on {dataset_name} after filtering rare classes.')

    try:
        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y, test_size=0.4, random_state=42, stratify=y
        )
    except ValueError:
        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y, test_size=0.4, random_state=42, stratify=None
        )

    try:
        X_val, X_test, y_val, y_test = train_test_split(
            X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
        )
    except ValueError:
        X_val, X_test, y_val, y_test = train_test_split(
            X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=None
        )

    return {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        'class_names': le.classes_,
        'num_classes': len(le.classes_),
    }


def _train_and_eval_config(data_dict, params, device, seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_graph = adaptive_graph_construction(
        data_dict['X_train'].values,
        data_dict['y_train'].values,
        adaptive_metric='euclidean',
        threshold=params['threshold']
    )
    val_graph = adaptive_graph_construction(
        data_dict['X_val'].values,
        data_dict['y_val'].values,
        adaptive_metric='euclidean',
        threshold=params['threshold']
    )

    train_graph.train_mask = torch.ones(train_graph.num_nodes, dtype=torch.bool)
    val_graph.test_mask = torch.ones(val_graph.num_nodes, dtype=torch.bool)

    train_graph = train_graph.to(device)
    val_graph = val_graph.to(device)

    output_dim = 1 if data_dict['num_classes'] == 2 else data_dict['num_classes']
    model = GCN(train_graph.x.shape[1], hidden_dim=params['hidden_dim'], output_dim=output_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

    model.train()
    for _ in range(params['epochs']):
        optimizer.zero_grad()
        logits = model(train_graph.x, train_graph.edge_index)
        if data_dict['num_classes'] == 2:
            loss = F.binary_cross_entropy_with_logits(logits.squeeze(), train_graph.y.float())
        else:
            loss = F.cross_entropy(logits, train_graph.y.long())
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(val_graph.x, val_graph.edge_index)
        y_true = val_graph.y.detach().cpu().numpy()

        if data_dict['num_classes'] == 2:
            y_pred = (torch.sigmoid(val_logits).detach().cpu().numpy().ravel() > 0.5).astype(int)
        else:
            y_pred = torch.softmax(val_logits, dim=1).argmax(dim=1).detach().cpu().numpy()

    val_acc = accuracy_score(y_true, y_pred)
    val_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return val_f1, val_acc


def _vector_to_params(vector):
    return {
        'hidden_dim': int(np.clip(round(vector[0]), 32, 256)),
        'lr': float(np.clip(vector[1], 1e-4, 1e-2)),
        'weight_decay': float(np.clip(vector[2], 1e-6, 1e-2)),
        'epochs': int(np.clip(round(vector[3]), 50, 250)),
        'threshold': float(np.clip(vector[4], 0.1, 0.9)),
    }


def _levy_step(beta=1.5, size=1):
    sigma = (
        math.gamma(1 + beta) * np.sin(np.pi * beta / 2)
        / (math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2)))
    ) ** (1 / beta)
    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)
    return u / (np.abs(v) ** (1 / beta) + 1e-9)


def secretary_bird_optimization(objective_fn, bounds, population_size=8, iterations=8, seed=42):
    np.random.seed(seed)

    dim = len(bounds)
    lower = np.array([b[0] for b in bounds], dtype=float)
    upper = np.array([b[1] for b in bounds], dtype=float)

    population = np.random.uniform(lower, upper, size=(population_size, dim))
    fitness = np.array([objective_fn(ind) for ind in population])

    best_idx = int(np.argmax(fitness))
    best_pos = population[best_idx].copy()
    best_fit = float(fitness[best_idx])

    history = []

    for iteration in range(1, iterations + 1):
        for i in range(population_size):
            current = population[i].copy()
            r1, r2 = np.random.rand(), np.random.rand()
            A = 2 * r1 - 1
            C = 2 * r2

            if np.random.rand() < 0.5:
                random_peer = population[np.random.randint(0, population_size)]
                candidate = current + A * (C * random_peer - current)
            else:
                levy = _levy_step(size=dim)
                candidate = best_pos + 0.05 * levy * (best_pos - current)

            candidate = np.clip(candidate, lower, upper)
            candidate_fit = objective_fn(candidate)

            if candidate_fit > fitness[i]:
                population[i] = candidate
                fitness[i] = candidate_fit

                if candidate_fit > best_fit:
                    best_fit = float(candidate_fit)
                    best_pos = candidate.copy()

        history.append({'iteration': iteration, 'best_macro_f1': best_fit})
        print(f"Iter {iteration:02d} | Best Macro-F1: {best_fit:.4f}")

    return best_pos, best_fit, pd.DataFrame(history)


def run_sbo_for_dataset(
    dataset_name,
    run_sbo=False,
    sample_size=4000,
    population_size=6,
    iterations=6,
    eval_repeats=3,
    seed=42,
):
    if not run_sbo:
        print(f"SBO for {dataset_name} is configured but not executed. Set run_sbo=True.")
        return None, None

    print(f"Running SBO hyperparameter optimization for: {dataset_name}")
    hpo_data = prepare_dataset_for_sbo(
        dataset_name=dataset_name,
        datasets_dict=datasets,
        target_columns_dict=target_columns,
        sample_size=sample_size,
    )

    bounds = [
        (32, 256),
        (1e-4, 1e-2),
        (1e-6, 1e-2),
        (50, 250),
        (0.1, 0.9),
    ]

    default_params = {'hidden_dim': 64, 'lr': 0.001, 'weight_decay': 0.0, 'epochs': 300, 'threshold': 0.5}
    baseline_scores = []
    for rep in range(eval_repeats):
        baseline_f1, _ = _train_and_eval_config(hpo_data, default_params, device, seed=seed + rep)
        baseline_scores.append(baseline_f1)
    baseline_macro_f1 = float(np.mean(baseline_scores))
    print(f"Baseline macro-F1 over {eval_repeats} run(s): {baseline_macro_f1:.4f}")

    def sbo_objective(vector):
        params = _vector_to_params(vector)
        scores = []
        for rep in range(eval_repeats):
            macro_f1, _ = _train_and_eval_config(hpo_data, params, device, seed=seed + rep)
            scores.append(macro_f1)
        return float(np.mean(scores))

    best_vector, best_score, sbo_history = secretary_bird_optimization(
        objective_fn=sbo_objective,
        bounds=bounds,
        population_size=population_size,
        iterations=iterations,
        seed=seed,
    )

    best_params = _vector_to_params(best_vector)

    if 'BEST_BG_GCN_PARAMS_BY_DATASET' not in globals():
        globals()['BEST_BG_GCN_PARAMS_BY_DATASET'] = {}

    BEST_BG_GCN_PARAMS_BY_DATASET[dataset_name] = best_params.copy()
    print(f"Saved BEST_BG_GCN_PARAMS_BY_DATASET['{dataset_name}']: {best_params}")
    print(f"Best Validation Macro-F1: {best_score:.4f}")
    print(f"Improvement over baseline: {best_score - baseline_macro_f1:+.4f}")
    display(sbo_history)

    return best_params, sbo_history

In [11]:
# Sequential comparison pipeline: baseline (default) -> HPO -> tuned experiment (dataset-by-dataset)
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

# =========================
# Configuration
# =========================
NUM_DATASETS_TO_TEST = 'all'   # valid: 1, 2, or 'all'
DATASET_ORDER = ['CICIDS2017', 'NSL-KDD', 'UAVIDS']
RUN_HPO_IN_THIS_PIPELINE = True
MIN_HPO_GAIN_TO_KEEP = 0.0  # keep tuned params only if validation gain >= this

BASELINE_BG_GCN_PARAMS = {
    'hidden_dim': 64,
    'lr': 0.001,
    'weight_decay': 0.0,
    'epochs': 300,
    'threshold': 0.5,
}

HPO_CFG = {
    'sample_size': 4000,
    'population_size': 6,
    'iterations': 6,
    'eval_repeats': 3,
    'seed': 42,
}

EXPERIMENT_CFG = {
    'new_dataset_size': 5000,
    'random_state': 42,
}


def _select_dataset_names(order, how_many):
    if str(how_many).lower() == 'all':
        return order[:]
    n = int(how_many)
    if n not in (1, 2, len(order)):
        raise ValueError(f'NUM_DATASETS_TO_TEST must be 1, 2, or \'all\'. Got: {how_many}')
    return order[:n]


def _create_imbalanced_subset(df, target_col, new_dataset_size=5000, random_state=42):
    class_counts = df[target_col].value_counts()
    large_classes = class_counts[class_counts > 500]
    small_classes = class_counts[class_counts <= 500]

    total_large_samples = max(1, large_classes.sum())
    scaled_counts = (large_classes / total_large_samples * new_dataset_size).astype(int)

    sampled_data = []
    for class_label, original_count in large_classes.items():
        sample_size = min(int(scaled_counts[class_label]), int(original_count))
        sample_size = max(1, sample_size)
        sampled_class = df[df[target_col] == class_label].sample(n=sample_size, random_state=random_state)
        sampled_data.append(sampled_class)

    for class_label in small_classes.index:
        sampled_data.append(df[df[target_col] == class_label])

    return pd.concat(sampled_data).reset_index(drop=True)


def _prepare_dataset_for_experiment(dataset_name, model_hparams, new_dataset_size=5000, random_state=42):
    target_col = target_columns[dataset_name]
    temp = datasets[dataset_name].copy()
    original_target = temp[target_col].copy()

    df = preprocess_data(temp)
    df[target_col] = original_target.iloc[:len(df)].reset_index(drop=True)

    if dataset_name == 'NSL-KDD':
        if '_map_nsl_attack_group' in globals():
            df[target_col] = df[target_col].apply(_map_nsl_attack_group)
        else:
            dos_attacks = {'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop', 'apache2', 'mailbomb', 'processtable', 'udpstorm'}
            probing_attacks = {'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint'}
            u2r_attacks = {'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack', 'xterm', 'ps'}
            r2l_attacks = {'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy', 'warezclient', 'warezmaster', 'httptunnel', 'named', 'sendmail', 'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop'}
            def _fallback_map(label):
                label = str(label).strip()
                if label == 'normal':
                    return 'normal'
                if label in probing_attacks:
                    return 'probing'
                if label in dos_attacks:
                    return 'DoS'
                if label in u2r_attacks:
                    return 'U2R'
                if label in r2l_attacks:
                    return 'R2L'
                return np.nan
            df[target_col] = df[target_col].apply(_fallback_map)

    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    numeric_cols_df = df.select_dtypes(include=['number']).columns
    df[numeric_cols_df] = df[numeric_cols_df].fillna(df[numeric_cols_df].median())

    le = LabelEncoder()
    df[target_col] = le.fit_transform(df[target_col].astype(str))
    class_names = le.classes_

    df = _create_imbalanced_subset(
        df,
        target_col=target_col,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    X = df.drop(columns=[target_col])
    y = df[target_col]

    numeric_cols = X.select_dtypes(include=['number']).columns
    scaler = StandardScaler().fit(X[numeric_cols])
    X = pd.DataFrame(scaler.transform(X[numeric_cols]))

    train_X, test_X, train_y, test_y = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    def introduce_feature_correlation(X_local, correlation_level=0.9):
        num_features = X_local.shape[1]
        correlation_matrix = np.random.uniform(correlation_level, 1.0, (num_features, num_features))
        return np.dot(X_local, correlation_matrix)

    train_X1 = introduce_feature_correlation(train_X, correlation_level=0.9)
    test_X1 = introduce_feature_correlation(test_X, correlation_level=0.9)

    def drop_strong_features(X_local, y_local, keep_ratio=0.3):
        mi = mutual_info_classif(X_local, y_local)
        important_features = np.argsort(mi)[-int(len(mi) * keep_ratio):]
        return X_local[:, important_features]

    train_X1 = drop_strong_features(train_X1, train_y, keep_ratio=0.3)
    test_X1 = drop_strong_features(test_X1, test_y, keep_ratio=0.3)

    train_X1 = pd.DataFrame(train_X1)
    test_X1 = pd.DataFrame(test_X1)

    graph_threshold = float(model_hparams.get('threshold', 0.5))
    train_graph = adaptive_graph_construction(
        train_X.values, train_y.values, adaptive_metric='euclidean', threshold=graph_threshold
    )
    test_graph = adaptive_graph_construction(
        test_X.values, test_y.values, adaptive_metric='euclidean', threshold=graph_threshold
    )

    train_graph.train_mask = torch.zeros(len(train_y), dtype=torch.bool)
    train_graph.train_mask[:int(0.9 * len(train_y))] = True
    test_graph.test_mask = torch.zeros(test_graph.num_nodes, dtype=torch.bool)

    train_graph.x = train_graph.x.to(device)
    train_graph.edge_index = train_graph.edge_index.to(device)
    train_graph.y = train_graph.y.to(device)

    test_graph.x = test_graph.x.to(device)
    test_graph.edge_index = test_graph.edge_index.to(device)
    test_graph.y = test_graph.y.to(device)

    return {
        'train_X1': train_X1,
        'test_X1': test_X1,
        'train_y': train_y,
        'test_y': test_y,
        'train_graph': train_graph,
        'test_graph': test_graph,
        'class_names': class_names,
    }


def run_single_dataset_experiment(dataset_name, model_hparams, new_dataset_size=5000, random_state=42):
    bundle = _prepare_dataset_for_experiment(
        dataset_name=dataset_name,
        model_hparams=model_hparams,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    results_df, confusion_matrices, split_metrics_tables = run_all_benchmarks(
        bundle['train_X1'],
        bundle['test_X1'],
        bundle['train_y'],
        bundle['test_y'],
        bundle['train_graph'],
        bundle['test_graph'],
        k=10,
        class_weights=None,
        model_hparams=model_hparams,
    )

    model_row = results_df.loc[results_df['Model'] == 'BG-GCN'].iloc[0]
    test_macro_f1 = float(model_row['F1'])
    test_accuracy = float(model_row['Accuracy'])

    payload = {
        'confusion_matrices': confusion_matrices,
        'class_names': bundle['class_names'],
        'split_metrics_tables': split_metrics_tables,
    }

    del bundle
    torch.cuda.empty_cache()
    gc.collect()

    return {
        'results_df': results_df,
        'payload': payload,
        'test_macro_f1': test_macro_f1,
        'test_accuracy': test_accuracy,
    }


selected_datasets = _select_dataset_names(DATASET_ORDER, NUM_DATASETS_TO_TEST)
print(f'Selected datasets: {selected_datasets}')

if 'BEST_BG_GCN_PARAMS_BY_DATASET' not in globals():
    BEST_BG_GCN_PARAMS_BY_DATASET = {}

dataset_comparison_rows = []
sequential_comparison_payload = {}

for idx, dataset_name in enumerate(selected_datasets, start=1):
    print(f"\n{'='*95}")
    print(f"DATASET {idx}/{len(selected_datasets)}: {dataset_name}")
    print(f"{'='*95}")

    print('\n[Step A] Experiment WITHOUT HPO (default params)')
    baseline_out = run_single_dataset_experiment(
        dataset_name=dataset_name,
        model_hparams=BASELINE_BG_GCN_PARAMS,
        new_dataset_size=EXPERIMENT_CFG['new_dataset_size'],
        random_state=EXPERIMENT_CFG['random_state'],
    )
    display(baseline_out['results_df'])

    chosen_params = BASELINE_BG_GCN_PARAMS.copy()
    hpo_history = None
    hpo_keep = False
    hpo_val_gain = np.nan
    hpo_test_gain = np.nan

    if RUN_HPO_IN_THIS_PIPELINE:
        print('\n[Step B] Run HPO')
        best_params, hpo_history = run_sbo_for_dataset(
            dataset_name=dataset_name,
            run_sbo=True,
            sample_size=HPO_CFG['sample_size'],
            population_size=HPO_CFG['population_size'],
            iterations=HPO_CFG['iterations'],
            eval_repeats=HPO_CFG['eval_repeats'],
            seed=HPO_CFG['seed'] + idx - 1,
        )
        if best_params is not None:
            try:
                hpo_val_gain = float(hpo_history['best_macro_f1'].max()) - float(hpo_history['best_macro_f1'].iloc[0])
            except Exception:
                hpo_val_gain = np.nan

            tuned_probe = run_single_dataset_experiment(
                dataset_name=dataset_name,
                model_hparams=best_params,
                new_dataset_size=EXPERIMENT_CFG['new_dataset_size'],
                random_state=EXPERIMENT_CFG['random_state'],
            )
            hpo_test_gain = tuned_probe['test_macro_f1'] - baseline_out['test_macro_f1']

            if (np.isnan(hpo_val_gain) or hpo_val_gain >= MIN_HPO_GAIN_TO_KEEP) and (hpo_test_gain > 0):
                chosen_params = best_params.copy()
                hpo_keep = True
                tuned_out = tuned_probe
                print(f"Keeping HPO params for {dataset_name}: test Macro-F1 gain {hpo_test_gain:+.4f}")
            else:
                print(f"Discarding HPO params for {dataset_name}: validation gain {hpo_val_gain:+.4f}, test gain {hpo_test_gain:+.4f}")
                tuned_out = baseline_out
        else:
            tuned_out = baseline_out
    else:
        print('\n[Step B] HPO skipped by configuration (RUN_HPO_IN_THIS_PIPELINE=False)')
        tuned_out = baseline_out

    if not RUN_HPO_IN_THIS_PIPELINE:
        print('\n[Step C] Selected params = baseline (HPO skipped)')
    elif hpo_keep:
        print('\n[Step C] Experiment WITH HPO-selected params (kept)')
    else:
        print('\n[Step C] Experiment WITH baseline params (HPO discarded)')

    display(tuned_out['results_df'])

    abs_gain = tuned_out['test_macro_f1'] - baseline_out['test_macro_f1']
    rel_gain_pct = (abs_gain / baseline_out['test_macro_f1'] * 100.0) if baseline_out['test_macro_f1'] > 0 else np.nan

    print(
        f"[Improvement] {dataset_name} | Test Macro-F1: "
        f"baseline={baseline_out['test_macro_f1']:.4f}, selected={tuned_out['test_macro_f1']:.4f}, "
        f"delta={abs_gain:+.4f} ({rel_gain_pct:+.2f}%) | hpo_kept={hpo_keep}"
    )

    dataset_comparison_rows.append({
        'Dataset': dataset_name,
        'Baseline Params': str(BASELINE_BG_GCN_PARAMS),
        'Selected Params': str(chosen_params),
        'HPO Params Kept': hpo_keep,
        'HPO Validation Gain Proxy': hpo_val_gain,
        'HPO Test Gain Probe': hpo_test_gain,
        'Baseline Test Accuracy': baseline_out['test_accuracy'],
        'Selected Test Accuracy': tuned_out['test_accuracy'],
        'Accuracy Gain': tuned_out['test_accuracy'] - baseline_out['test_accuracy'],
        'Baseline Test Macro-F1': baseline_out['test_macro_f1'],
        'Selected Test Macro-F1': tuned_out['test_macro_f1'],
        'Macro-F1 Gain': abs_gain,
        'Macro-F1 Gain (%)': rel_gain_pct,
    })

    sequential_comparison_payload[dataset_name] = {
        'baseline': baseline_out['payload'],
        'selected': tuned_out['payload'],
        'hpo_history': hpo_history,
        'hpo_kept': hpo_keep,
    }

SEQUENTIAL_HPO_COMPARISON_DF = pd.DataFrame(dataset_comparison_rows)
SEQUENTIAL_HPO_COMPARISON_PAYLOAD = sequential_comparison_payload

print('\nFinal summary: baseline vs selected (after HPO gate)')
display(SEQUENTIAL_HPO_COMPARISON_DF)

Selected datasets: ['CICIDS2017', 'NSL-KDD', 'UAVIDS']

DATASET 1/3: CICIDS2017

[Step A] Experiment WITHOUT HPO (default params)

--- Training BG-GCN ---

Benchmark Results:
| Model   |   Accuracy |      AUC |   Precision |   Recall |       F1 | Time (s)   | Memory (MB)   |
|:--------|-----------:|---------:|------------:|---------:|---------:|:-----------|:--------------|
| BG-GCN  |   0.993014 | 0.980935 |    0.759632 | 0.770072 | 0.764683 | 7.07 s     | 0.36 MB       |


,Model,Accuracy,AUC,Precision,Recall,F1,Time (s),Memory (MB)
0,BG-GCN,0.993014,0.980935,0.759632,0.770072,0.764683,7.07 s,0.36 MB



[Step B] Run HPO
Running SBO hyperparameter optimization for: CICIDS2017
Baseline macro-F1 over 3 run(s): 0.9453
Iter 01 | Best Macro-F1: 0.5792
Iter 02 | Best Macro-F1: 0.5809
Iter 03 | Best Macro-F1: 0.5821
Iter 04 | Best Macro-F1: 0.5821
Iter 05 | Best Macro-F1: 0.5822
Iter 06 | Best Macro-F1: 0.5822
Saved BEST_BG_GCN_PARAMS_BY_DATASET['CICIDS2017']: {'hidden_dim': 169, 'lr': 0.0014804465232980135, 'weight_decay': 0.0029324459836725545, 'epochs': 124, 'threshold': 0.4649413141758766}
Best Validation Macro-F1: 0.5822
Improvement over baseline: -0.3631


,iteration,best_macro_f1
0,1,0.579162
1,2,0.580881
2,3,0.582089
3,4,0.582089
4,5,0.582196
5,6,0.582196



--- Training BG-GCN ---

Benchmark Results:
| Model   |   Accuracy |      AUC |   Precision |   Recall |      F1 | Time (s)   | Memory (MB)   |
|:--------|-----------:|---------:|------------:|---------:|--------:|:-----------|:--------------|
| BG-GCN  |    0.96507 | 0.961528 |    0.320059 | 0.332808 | 0.32628 | 26.05 s    | 0.36 MB       |
Discarding HPO params for CICIDS2017: validation gain +0.0030, test gain -0.4384

[Step C] Experiment WITH baseline params (HPO discarded)


...

,iteration,best_macro_f1
0,1,0.944049
1,2,0.944049
2,3,0.944049
3,4,0.944049
4,5,0.944049
5,6,0.944049



--- Training BG-GCN ---

Benchmark Results:
| Model   |   Accuracy |      AUC |   Precision |   Recall |       F1 | Time (s)   | Memory (MB)   |
|:--------|-----------:|---------:|------------:|---------:|---------:|:-----------|:--------------|
| BG-GCN  |      0.976 | 0.999237 |    0.975951 | 0.974786 | 0.975315 | 36.71 s    | 0.35 MB       |
Keeping HPO params for UAVIDS: test Macro-F1 gain +0.0257

[Step C] Experiment WITH HPO-selected params (kept)


,Model,Accuracy,AUC,Precision,Recall,F1,Time (s),Memory (MB)
0,BG-GCN,0.976,0.999237,0.975951,0.974786,0.975315,36.71 s,0.35 MB


[Improvement] UAVIDS | Test Macro-F1: baseline=0.9496, selected=0.9753, delta=+0.0257 (+2.71%) | hpo_kept=True

Final summary: baseline vs selected (after HPO gate)


,Dataset,Baseline Params,Selected Params,HPO Params Kept,HPO Validation Gain Proxy,HPO Test Gain Probe,Baseline Test Accuracy,Selected Test Accuracy,Accuracy Gain,Baseline Test Macro-F1,Selected Test Macro-F1,Macro-F1 Gain,Macro-F1 Gain (%)
0,CICIDS2017,"{'hidden_dim': 64, 'lr': 0.001, 'weight_decay'...","{'hidden_dim': 64, 'lr': 0.001, 'weight_decay'...",False,0.003035,-0.438403,0.993014,0.993014,0.000,0.764683,0.764683,0.000000,0.000000
1,NSL-KDD,"{'hidden_dim': 64, 'lr': 0.001, 'weight_decay'...","{'hidden_dim': 64, 'lr': 0.001, 'weight_decay'...",False,0.000000,-0.018103,0.979492,0.979492,0.000,0.877270,0.877270,0.000000,0.000000
2,UAVIDS,"{'hidden_dim': 64, 'lr': 0.001, 'weight_decay'...","{'hidden_dim': 181, 'lr': 0.008640475926765897...",True,0.000000,0.025717,0.951000,0.976000,0.025,0.949597,0.975315,0.025717,2.708228


In [ ]:
# Compact bar chart: baseline vs tuned Macro-F1 per selected dataset
if 'SEQUENTIAL_HPO_COMPARISON_DF' not in globals() or SEQUENTIAL_HPO_COMPARISON_DF.empty:
    print('Run the sequential comparison pipeline cell first to generate SEQUENTIAL_HPO_COMPARISON_DF.')
else:
    plot_df = SEQUENTIAL_HPO_COMPARISON_DF.copy()
    x = np.arange(len(plot_df))
    width = 0.36

    plt.figure(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars_baseline = plt.bar(
        x - width / 2,
        plot_df['Baseline Test Macro-F1'].values,
        width=width,
        label='Baseline (Default)',
        color='#9E9E9E',
    )
    bars_tuned = plt.bar(
        x + width / 2,
        plot_df['Tuned Test Macro-F1'].values,
        width=width,
        label='Tuned (After HPO)',
        color='#2E7D32',
    )

    for i, row in plot_df.reset_index(drop=True).iterrows():
        gain = float(row['Macro-F1 Gain'])
        gain_pct = float(row['Macro-F1 Gain (%)']) if pd.notna(row['Macro-F1 Gain (%)']) else np.nan
        y_top = max(float(row['Baseline Test Macro-F1']), float(row['Tuned Test Macro-F1']))
        label = f"{gain:+.4f}" if np.isnan(gain_pct) else f"{gain:+.4f} ({gain_pct:+.2f}%)"
        plt.text(i, min(1.05, y_top + 0.02), label, ha='center', va='bottom', fontsize=8, color='black')

    for bars in [bars_baseline, bars_tuned]:
        for bar in bars:
            h = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                min(1.03, h + 0.01),
                f"{h:.3f}",
                ha='center',
                va='bottom',
                fontsize=7,
            )

    plt.xticks(x, plot_df['Dataset'].tolist())
    plt.ylim(0, 1.08)
    plt.ylabel('Test Macro-F1')
    plt.xlabel('Dataset')
    plt.title('Baseline vs Tuned (After HPO): Test Macro-F1')
    plt.grid(axis='y', alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Compact bar chart: baseline vs tuned Accuracy per selected dataset
if 'SEQUENTIAL_HPO_COMPARISON_DF' not in globals() or SEQUENTIAL_HPO_COMPARISON_DF.empty:
    print('Run the sequential comparison pipeline cell first to generate SEQUENTIAL_HPO_COMPARISON_DF.')
else:
    plot_df = SEQUENTIAL_HPO_COMPARISON_DF.copy()
    x = np.arange(len(plot_df))
    width = 0.36

    plt.figure(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars_baseline = plt.bar(
        x - width / 2,
        plot_df['Baseline Test Accuracy'].values,
        width=width,
        label='Baseline (Default)',
        color='#9E9E9E',
    )
    bars_tuned = plt.bar(
        x + width / 2,
        plot_df['Tuned Test Accuracy'].values,
        width=width,
        label='Tuned (After HPO)',
        color='#1565C0',
    )

    for i, row in plot_df.reset_index(drop=True).iterrows():
        gain = float(row['Accuracy Gain'])
        base = float(row['Baseline Test Accuracy'])
        gain_pct = (gain / base * 100.0) if base > 0 else np.nan
        y_top = max(float(row['Baseline Test Accuracy']), float(row['Tuned Test Accuracy']))
        label = f"{gain:+.4f}" if np.isnan(gain_pct) else f"{gain:+.4f} ({gain_pct:+.2f}%)"
        plt.text(i, min(1.05, y_top + 0.02), label, ha='center', va='bottom', fontsize=8, color='black')

    for bars in [bars_baseline, bars_tuned]:
        for bar in bars:
            h = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                min(1.03, h + 0.01),
                f"{h:.3f}",
                ha='center',
                va='bottom',
                fontsize=7,
            )

    plt.xticks(x, plot_df['Dataset'].tolist())
    plt.ylim(0, 1.08)
    plt.ylabel('Test Accuracy')
    plt.xlabel('Dataset')
    plt.title('Baseline vs Tuned (After HPO): Test Accuracy')
    plt.grid(axis='y', alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()